[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Changing a Schema &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/shipped.db` as the notebook's Setup did, and defines what the worked
examples defined: `copy_of_shipped`, `open_database`, `create_statement`, `schema_change`,
`rebuild`, the two new table definitions, the three migrations and `migrate`. Run it first. Every
task works on a copy of its own, so the tasks do not depend on one another, and the last cell
removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
from contextlib import contextmanager
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
SHIPPED = SCRATCH / "shipped.db"
SHIPPED.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}
CODES = {"Bergen": "BGO", "Oslo": "OSL", "Svalbard": "LYR", "Tromso": "TOS", "Kirkenes": "KKN"}


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(SHIPPED)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
    CREATE INDEX readings_by_station_hour ON readings (station_id, hour);
    CREATE VIEW daily_means AS
        SELECT station_id, date(hour) AS day, ROUND(AVG(celsius), 1) AS mean
        FROM readings
        GROUP BY station_id, date(hour);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()


def copy_of_shipped(name):
    """A fresh copy of the database as it shipped, at scratch/<name>.db."""
    path = SCRATCH / f"{name}.db"
    shutil.copy(SHIPPED, path)
    return path


def open_database(path):
    """A connection that enforces foreign keys, with autocommit=True, so that the SQL writes its own transactions."""
    conn = sqlite3.connect(path, autocommit=True)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def create_statement(conn, name):
    """The CREATE statement SQLite keeps for a table, an index, a view or a trigger."""
    return conn.execute("SELECT sql FROM sqlite_schema WHERE name = ?", (name,)).fetchone()[0]


@contextmanager
def schema_change(conn):
    """Steps 1, 2 and 10 to 12: foreign keys off, one transaction, and a commit only if no foreign key is broken."""
    conn.execute("PRAGMA foreign_keys = OFF")
    conn.execute("BEGIN")
    try:
        yield
        broken = conn.execute("PRAGMA foreign_key_check").fetchall()
        if broken:
            raise sqlite3.IntegrityError(f"{len(broken)} rows break a foreign key, the first {broken[0]}")
        conn.execute("COMMIT")
    except BaseException:
        conn.execute("ROLLBACK")
        raise
    finally:
        conn.execute("PRAGMA foreign_keys = ON")


def rebuild(conn, table, create_new):
    """Steps 3 to 9: rebuild a table from create_new, which creates <table>_new, keeping its rows, indexes and views."""
    kept = [sql for (sql,) in conn.execute(
        "SELECT sql FROM sqlite_schema WHERE tbl_name = ? AND type IN ('index', 'trigger') AND sql IS NOT NULL", (table,))]
    views = conn.execute("SELECT name, sql FROM sqlite_schema WHERE type = 'view'").fetchall()
    for name, _ in views:
        conn.execute(f'DROP VIEW "{name}"')
    conn.execute(create_new)
    old_columns = {column[1] for column in conn.execute(f'PRAGMA table_info("{table}")')}
    columns = ", ".join(f'"{column[1]}"' for column in conn.execute(f'PRAGMA table_info("{table}_new")')
                        if column[1] in old_columns)
    conn.execute(f'INSERT INTO "{table}_new" ({columns}) SELECT {columns} FROM "{table}"')
    conn.execute(f'DROP TABLE "{table}"')
    conn.execute(f'ALTER TABLE "{table}_new" RENAME TO "{table}"')
    for sql in kept + [sql for _, sql in views]:
        conn.execute(sql)


NEW_READINGS = """
    CREATE TABLE readings_new (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL REFERENCES stations (id),
        hour       TEXT NOT NULL,
        celsius    REAL CONSTRAINT plausible_celsius CHECK (celsius BETWEEN -90 AND 60),
        UNIQUE (station_id, hour)
    ) STRICT
"""
NEW_STATIONS = """
    CREATE TABLE stations_new (
        id       INTEGER PRIMARY KEY,
        name     TEXT NOT NULL UNIQUE,
        latitude REAL NOT NULL CONSTRAINT plausible_latitude CHECK (latitude BETWEEN -90 AND 90)
    ) STRICT
"""


def make_readings_strict(conn):
    """Version 1: readings STRICT, one reading for a station and hour, and plausible temperatures."""
    rebuild(conn, "readings", NEW_READINGS)


def make_stations_strict(conn):
    """Version 2: stations STRICT, with names of their own and latitudes on Earth."""
    rebuild(conn, "stations", NEW_STATIONS)


def add_station_codes(conn):
    """Version 3: a code for every station, different for every station."""
    conn.execute("ALTER TABLE stations ADD COLUMN code TEXT")
    conn.executemany("UPDATE stations SET code = ? WHERE name = ?", [(code, name) for name, code in CODES.items()])
    conn.execute("CREATE UNIQUE INDEX stations_by_code ON stations (code)")


MIGRATIONS = [make_readings_strict, make_stations_strict, add_station_codes]


def migrate(conn, migrations=MIGRATIONS):
    """Run every migration the database has not had, each in its own transaction with its number, and return the numbers."""
    version = conn.execute("PRAGMA user_version").fetchone()[0]
    applied = []
    for number in range(version + 1, len(migrations) + 1):
        with schema_change(conn):
            migrations[number - 1](conn)
            conn.execute(f"PRAGMA user_version = {number}")
        applied.append(number)
    return applied


print("built", SHIPPED)


built scratch/shipped.db


**1.** A column with a default for the rows already there.


In [2]:
conn = open_database(copy_of_shipped("elevation"))
conn.execute("ALTER TABLE stations ADD COLUMN elevation REAL NOT NULL DEFAULT 0")

print(create_statement(conn, "stations"))
print(conn.execute("SELECT * FROM stations WHERE name = 'Tromso'").fetchone())
conn.close()


CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL, elevation REAL NOT NULL DEFAULT 0)
(4, 'Tromso', 69.65, 0.0)


The column went on the end of the `CREATE` statement, and Tromso's row, written before the column
existed, reads back the default. `NOT NULL` needed the default: without it, the rows already there
would have read `NULL`.


**2.** What follows a table's rename.


In [3]:
conn = open_database(copy_of_shipped("observations"))
conn.execute("ALTER TABLE readings RENAME TO observations")

print(create_statement(conn, "daily_means"))
print()
print(create_statement(conn, "readings_by_station_hour"))
conn.close()


CREATE VIEW daily_means AS
        SELECT station_id, date(hour) AS day, ROUND(AVG(celsius), 1) AS mean
        FROM "observations"
        GROUP BY station_id, date(hour)

CREATE INDEX readings_by_station_hour ON "observations" (station_id, hour)


The view now reads from `"observations"`, and the index is now on it, still under its old name,
which the rename does not touch. A program that still queries `readings` would fail, which is why a
rename belongs in a migration that every copy of the file receives.


**3.** A rebuild the old rows cannot pass.


In [4]:
REQUIRED_CELSIUS = """
    CREATE TABLE readings_new (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL REFERENCES stations (id),
        hour       TEXT NOT NULL,
        celsius    REAL NOT NULL CONSTRAINT plausible_celsius CHECK (celsius BETWEEN -90 AND 60),
        UNIQUE (station_id, hour)
    ) STRICT
"""

conn = open_database(copy_of_shipped("required_celsius"))
try:
    with schema_change(conn):
        rebuild(conn, "readings", REQUIRED_CELSIUS)
except sqlite3.IntegrityError as error:
    print("refused:", error)

print("readings:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
print("STRICT:", conn.execute("SELECT strict FROM pragma_table_list WHERE name = 'readings'").fetchone()[0])
tables_and_views = "SELECT type, name FROM sqlite_schema WHERE type IN ('table', 'view') ORDER BY name"
print("tables and views:", conn.execute(tables_and_views).fetchall())
conn.close()


refused: NOT NULL constraint failed: readings_new.celsius
readings: 35040
STRICT: 0
tables and views: [('view', 'daily_means'), ('table', 'readings'), ('table', 'stations')]


The first of Svalbard's missing readings failed `NOT NULL` at the copy in step 5, and
`schema_change` rolled everything back: the view that `rebuild` had dropped is back, `readings_new`
is gone, and `readings` is the flexible table it was, with every row. A rule the old rows break
needs those rows dealt with first, in the same migration or an earlier one.


**4.** A database newer than the program.


In [5]:
def careful_migrate(conn, migrations=MIGRATIONS):
    """migrate, after refusing a database that has had migrations this program does not know."""
    version = conn.execute("PRAGMA user_version").fetchone()[0]
    if version > len(migrations):
        raise RuntimeError(f"the database is at version {version}, "
                           f"and this program knows migrations up to {len(migrations)}")
    return migrate(conn, migrations)


conn = open_database(copy_of_shipped("from_the_future"))
conn.execute("PRAGMA user_version = 9")
try:
    careful_migrate(conn)
except RuntimeError as error:
    print("refused:", error)
conn.close()


refused: the database is at version 9, and this program knows migrations up to 3


`migrate` alone would have found nothing to run and returned an empty list, and an old program would
have gone on writing to a schema it does not know. Refusing is the safe choice: a file at version 9
needs the program that made it.


**5.** A fourth migration.


In [6]:
def add_reading_sources(conn):
    """Version 4: where every reading came from, 'station' for the readings already there."""
    conn.execute("ALTER TABLE readings ADD COLUMN source TEXT NOT NULL DEFAULT 'station'")


conn = open_database(copy_of_shipped("four_migrations"))
print("applied:", migrate(conn, MIGRATIONS + [add_reading_sources]))
print("version:", conn.execute("PRAGMA user_version").fetchone()[0])
print(conn.execute("SELECT id, station_id, hour, source FROM readings ORDER BY id LIMIT 1").fetchone())
conn.close()


applied: [1, 2, 3, 4]
version: 4
(1, 1, '2025-01-01T00:00', 'station')


The new migration went on the end of the list, so a copy at version 3 would run only migration 4,
and a copy of the shipped database ran all four, in order. `ADD COLUMN` works on a STRICT table as on
any other, with a type the column must keep.


**6.** A broken foreign key, found.


In [7]:
conn = open_database(copy_of_shipped("orphan"))
conn.execute("PRAGMA foreign_keys = OFF")
conn.execute("INSERT INTO readings (station_id, hour, celsius) VALUES (99, '2026-01-01T00:00', 2.5)")

print(conn.execute("PRAGMA foreign_key_check").fetchall())
conn.close()


[('readings', 35041, 'stations', 0)]


With foreign keys off, the reading went in. `PRAGMA foreign_key_check` works whether foreign keys are
on or off, and reports every broken row as the table it is in, its rowid, the table it should point
at, and which of the table's foreign keys it breaks, counted from 0.

Last, remove the scratch folder:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Changing a Schema](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/13-changing-a-schema.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
